<a href="https://colab.research.google.com/github/Hejran-M/Adversarial-defense-for-battery-state-of-health-prediction-models/blob/main/1_CNN_to_estimate_SOH_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN

## 0. Import libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms,datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
from torch.utils.data import DataLoader, TensorDataset, random_split

import sys
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn import metrics

import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D  # Updated import statement
from tensorflow.keras.utils import plot_model
from keras import backend as K
from keras.models import load_model

## 1. Load dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/T25 (1).csv')

Mounted at /content/drive


In [3]:
set(df['Dataset'].tolist())

{'LFP_25C_0.5-1C_a',
 'LFP_25C_0.5-1C_b',
 'LFP_25C_0.5-1C_c',
 'LFP_25C_0.5-1C_d',
 'NCA_25C_0.5-1C_a',
 'NCA_25C_0.5-1C_b',
 'NCA_25C_0.5-1C_c',
 'NCA_25C_0.5-1C_d',
 'NMC_25C_0.5-1C_a',
 'NMC_25C_0.5-1C_b',
 'NMC_25C_0.5-1C_c',
 'NMC_25C_0.5-1C_d'}

In [4]:
df.columns

Index(['Unnamed: 0', 'Dataset', '#cycle', 'Capacity(dis)', 'SOH', 'RGB',
       'scaled_RGB', 'RGB_length', 'resized', 'resized_scaled'],
      dtype='object')

In [5]:
X = [eval(i) for i in df['resized_scaled'].tolist()]
X = np.array(X)
y = np.array(df['SOH'].tolist()) ##%changed

In [6]:
# labeling for stratified random sampling
label = df['SOH'].tolist()
label_l = []
for cnt in range(len(label)):
    i = label[cnt]
    if i>=1.0 :
        lab = '1~'
        num = 0
    elif 0.98<=i<1.0 :
        lab = '0.98~1'
        num = 1
    elif 0.96<=i<0.98 :
        lab = '0.96~0.98'
        num = 2
    elif 0.94<=i<0.96 :
        lab = '0.94~0.96'
        num = 3
    elif 0.92<=i<0.94 :
        lab = '0.92~0.94'
        num = 4
    elif 0.90<=i<0.92 :
        lab = '0.90~0.92'
        num = 5

    elif 0.88<=i<0.9 :
        lab = '0.88~0.9'
        num = 6
    elif 0.86<=i<0.88 :
        lab = '0.86~0.88'
        num = 7
    elif 0.84<=i<0.86 :
        lab = '0.84~0.86'
        num = 8
    elif 0.82<=i<0.84 :
        lab = '0.82~0.84'
        num = 9
    elif 0.80<=i<0.82 :
        lab = '0.80~0.82'
        num = 10


    elif 0.78<=i<0.8 :
        lab = '0.78~0.8'
        num = 11
    elif 0.76<=i<0.78 :
        lab = '0.76~0.78'
        num = 12
    elif 0.74<=i<0.76 :
        lab = '0.74~0.76'
        num = 13
    elif 0.72<=i<0.74 :
        lab = '0.72~0.74'
        num = 14
    elif 0.70<=i<0.72 :
        lab = '0.70~0.72'
        num = 15

    else:
        lab = '~0.7'
        num = 16
    label_l.append(num)

## 2. cnn regression

In [7]:
sss = StratifiedShuffleSplit(n_splits = 10, test_size = 0.3)
for train_idx, test_idx in sss.split(X, np.array(label_l)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = np.array(y)[train_idx], np.array(y)[test_idx]
    y_valid = np.array(label_l)[test_idx]

In [8]:
sss = StratifiedShuffleSplit(n_splits = 10, test_size = 0.3)
for train_idx, test_idx1 in sss.split(X_test, y_valid):
    X_valid, X_test_f = X_test[train_idx], X_test[test_idx1]
    y_valid, y_test_f = np.array(y_test)[train_idx], np.array(y_test)[test_idx1]
X_valid.shape

(3588, 128, 3)

In [9]:
data_list = []
for i in df['Dataset'][test_idx[test_idx1]].tolist():
    if 'LFP' in i:
        data_list.append('LFP')
    if 'NCA' in i:
        data_list.append('NCA')
    if 'NMC' in i:
        data_list.append('NMC')

In [10]:
x_train = X_train
x_valid = X_valid
x_test = X_test_f

In [11]:
rows = len(X[0])
cols = 3
input_shape = (rows, cols, 1)
x_train = x_train.reshape(x_train.shape[0], rows, cols, 1)
x_test = x_test.reshape(x_test.shape[0], rows, cols, 1)
x_valid = x_valid.reshape(x_valid.shape[0], rows, cols, 1)

x_train = x_train.astype('float32') /255.0
x_test = x_test.astype('float32')/255.0
x_valid = x_valid.astype('float32')/255.0

batch_size = 64
x_train.shape

(11963, 128, 3, 1)

In [12]:
###
model = Sequential()
model.add(Conv2D(16, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(Conv2D(32, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(Conv2D(64, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(MaxPooling2D(pool_size = (2,2), strides = (1,1)))
model.add(Flatten())
model.add(Dense(64,activation = 'relu'))
model.add(Dense(1))
model.summary()
###

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 3, 16)     │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 3, 32)     │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 3, 64)     │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 2, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16256)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,040,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,050,929 (4.01 MB)

 Trainable params: 1,050,929 (4.01 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
start = time.time()
epochs = 40
print('start time : ' , start)
model.compile(loss = 'mean_squared_error', optimizer = 'adam', metrics = ['mse'])
hist2 = model.fit(x_train, y_train, epochs = epochs, verbose = 1, validation_data = (x_valid, y_valid))

start time :  1776336161.6666846
Epoch 1/40
374/374 ━━━━━━━━━━━━━━━━━━━━ 28s 69ms/step - loss: 0.0126 - mse: 0.0126 - val_loss: 6.8776e-04 - val_mse: 6.8776e-04
Epoch 2/40
374/374 ━━━━━━━━━━━━━━━━━━━━ 23s 61ms/step - loss: 6.9662e-04 - mse: 6.9662e-04 - val_loss: 6.5330e-04 - val_mse: 6.5330e-04
Epoch 3/40
374/374 ━━━━━━━━━━━━━━━━━━━━ 25s 68ms/step - loss: 5.0715e-04 - mse: 5.0715e-04 - val_loss: 7.5048e-04 - val_mse: 7.5048e-04
Epoch 4/40
374/374 ━━━━━━━━━━━━━━━━━━━━ 41s 67ms/step - loss: 7.1095e-04 - mse: 7.1095e-04 - val_loss: 3.8425e-04 - val_mse: 3.8425e-04
Epoch 5/40
374/374 ━━━━━━━━━━━━━━━━━━━━ 23s 61ms/step - loss: 3.5011e-04 - mse: 3.5011e-04 - val_loss: 2.5106e-04 - val_mse: 2.5106e-04
Epoch 6/40
374/374 ━━━━━━━━━━━━━━━━━━━━ 24s 64ms/step - loss: 3.3029e-04 - mse: 3.3029e-04 - val_loss: 2.0510e-04 - val_mse: 2.0510e-04
Epoch 7/40
374/374 ━━━━━━━━━━━━━━━━━━━━ 23s 61ms/step - loss: 2.6685e-04 - mse: 2.6685e-04 - val_loss: 2.5658e-04 - val_mse: 2.5658e-04
Epoch 8/40


In [ ]:
y_vloss = hist2.history['val_loss']
y_loss = hist2.history['loss']
x_len = np.arange(len(y_loss))

plt.plot(x_len, y_vloss, marker='.', c='red', label="Validation-set Loss")
plt.plot(x_len, y_loss, marker='.', c='blue', label="Train-set Loss")
plt.legend()
plt.title('epoch40_loss')

In [ ]:
predict = model.predict(x_test, verbose = 0)

In [ ]:
p = []
t = []
for i in range(len(predict)):
    p.append(predict[i])
    t.append(y_test_f[i])
    #print('predicted SOH: ' , predict[i],',' , 'Answer SOH: '  , y_test[i])

plt.scatter(y_test_f, predict)
plt.plot(y_test_f, y_test_f)
plt.show()

In [ ]:
#model.save('/home/sbml/battery_rgb/220610/resize_not_intepolate/CNN/model_candidates/cnn4.h5')

In [ ]:
dic = {}
dic['x_test_SOH'] = t
dic['predict_SOH'] = p
dic['data'] = data_list
data = pd.DataFrame(dic)

In [ ]:
r2 = metrics.r2_score(y_test_f, predict)
rmse = metrics.mean_squared_error(y_test_f, predict)**0.5
mse = metrics.mean_squared_error(y_test_f, predict)
mae = metrics.mean_absolute_error(y_test_f, predict)
mape = metrics.mean_absolute_percentage_error(y_test_f,predict)

In [ ]:
dic = {}
dic['r2'] = [r2]
dic['rmse'] = [rmse]
dic['mse'] = [mse]
dic['mae'] = [mae]
dic['mape'] = [mape]
d = pd.DataFrame(dic)

In [ ]:
d